# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [2]:
# Write your code below.
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [4]:
import os
from glob import glob

# Write your code below.
from utils.logger import get_logger
_logs = get_logger(__name__)

PRICE_DATA = os.getenv("PRICE_DATA")
_logs.info(f"PRICE_DATA directory: {PRICE_DATA}")

parquet_files = glob(
    os.path.join(PRICE_DATA, "**", "*.parquet"),
    recursive=True,
)

_logs.info(f"Number of parquet files found: {len(parquet_files)}")

dd_price = dd.read_parquet(parquet_files)

2026-01-19 02:11:17,647, 3355711681.py, 9, INFO, PRICE_DATA directory: ../../05_src/data/prices/
2026-01-19 02:11:17,900, 3355711681.py, 16, INFO, Number of parquet files found: 3178


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [5]:
# Write your code below.
# Changing column name if it is needed
if "Adj Close" in dd_price.columns:
    _logs.info("Renaming column 'Adj Close' to 'Adj_Close'.")
    dd_price = dd_price.rename(columns={"Adj Close": "Adj_Close"})

import pandas as pd

dd_feat = (
    dd_price
        .set_index("ticker")
        .groupby("ticker", group_keys=False)
        .apply(
            lambda x: (
                x.sort_values("Date", ascending=True)
                 .assign(
                     Close_lag_1 = x["Close"].shift(1),
                     Adj_Close_lag_1 = x["Adj_Close"].shift(1),
                 )
                 .assign(
                     returns = lambda y: y["Close"] / y["Close_lag_1"] - 1,
                     hi_lo_range = lambda y: y["High"] - y["Low"],
                 )
            ),
            meta=pd.DataFrame(
                data={
                    "Date": "datetime64[ns]",
                    "Open": "f8",
                    "High": "f8",
                    "Low": "f8",
                    "Close": "f8",
                    "Adj_Close": "f8",
                    "Volume": "i8",
                    "source": "object",
                    "Year": "int32",
                    "Close_lag_1": "f8",
                    "Adj_Close_lag_1": "f8",
                    "returns": "f8",
                    "hi_lo_range": "f8",
                },
                index=pd.Index([], dtype=pd.StringDtype(), name="ticker"),
            ),
        )
)

_logs.info("Created feature dataframe dd_feat.")

2026-01-19 02:11:33,201, 487974657.py, 4, INFO, Renaming column 'Adj Close' to 'Adj_Close'.
2026-01-19 02:11:33,208, 487974657.py, 46, INFO, Created feature dataframe dd_feat.


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [6]:
# Write your code below.
# Convert Dask dataframe to Pandas
_logs.info("Converting dd_feat to pandas dataframe.")
feat_pd = dd_feat.compute()

_logs.info(f"Converted dataframe shape: {feat_pd.shape}")

# Add rolling 10-day moving average of returns
feat_pd = (
    feat_pd
        .reset_index()  # bring ticker back as a column
        .groupby("ticker", group_keys=False)
        .apply(
            lambda x: (
                x.sort_values("Date", ascending=True)
                 .assign(
                     returns_ma_10 = x["returns"].rolling(10).mean()
                 )
            )
        )
)

_logs.info("Added 10-day moving average of returns (returns_ma_10).")

2026-01-19 02:11:40,663, 2630096182.py, 3, INFO, Converting dd_feat to pandas dataframe.
2026-01-19 02:11:57,351, 2630096182.py, 6, INFO, Converted dataframe shape: (372301, 13)
C:\Users\kamoornani\AppData\Local\Temp\ipykernel_19980\2630096182.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
2026-01-19 02:11:57,517, 2630096182.py, 23, INFO, Added 10-day moving average of returns (returns_ma_10).


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?

> No. Dask supports rolling-window operations, so the moving average could have been computed directly in Dask.

+ Would it have been better to do it in Dask? Why?

> Yes for large datasets. Staying in Dask avoids loading all data into memory and preserves parallelism. Pandas is simpler and appropriate (and may even offer faster functions) when the data comfortably fits in RAM, but doing so sacrifices Dask’s parallelism.

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.